# FloodRisk · Auditoría inicial de datasets

En Colab: ejecuta las celdas de arriba hacia abajo. En la subida, selecciona el ZIP que contiene los CSV (o selecciona varios CSV). No necesitas GPU. RAR: extraer primero en tu equipo.

Esta revisión analiza todas las filas de los CSV cargados y muestra muestras de las excepciones. No entrena, elimina ni corrige datos. Los documentos incluidos son referencias, no instrucciones ejecutables.

Límites iniciales: 100 MB por archivo, 500 MB descomprimidos en total. El tamaño en memoria de texto puede ser mayor; si se supera, usar lectura por bloques en una siguiente versión.

## 1. Preparar el entorno

In [ ]:
import io
import json
import zipfile
from pathlib import PurePosixPath
import pandas as pd
from IPython.display import display

# Se conservan los valores originales como texto, incluidos los ceros de FIPS.
MAX_ARCHIVE_BYTES = 500 * 1024 * 1024
MAX_FILE_BYTES = 100 * 1024 * 1024
NA_MARKERS = {"", "na", "n/a", "nan", "null", "none"}
print("Entorno listo. pandas:", pd.__version__)


## 2. Subir los datos

La carga es temporal y se pierde al reiniciar el entorno. Puedes repetirla sin modificar los originales de tu equipo.

In [ ]:
from google.colab import files

print("Selecciona el ZIP del dataset o varios CSV. Si tienes RAR, extráelo primero en tu equipo.")
uploaded = files.upload()
if not uploaded:
    raise ValueError("No se seleccionaron archivos")
print("Archivos recibidos:", list(uploaded))


## 3. Definir las comprobaciones

Esta celda contiene las reglas automáticas. Solo define funciones; la siguiente las ejecuta.

In [ ]:
def unpack_inputs(uploaded):
    contents = {}
    inventory = []
    total_bytes = 0
    for filename, payload in uploaded.items():
        if filename.lower().endswith(".rar"):
            raise ValueError("Extrae el RAR en tu equipo y sube sus CSV o un ZIP.")
        if filename.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(payload)) as archive:
                members = [m for m in archive.infolist() if not m.is_dir()
                           and "__MACOSX" not in PurePosixPath(m.filename).parts
                           and not PurePosixPath(m.filename).name.startswith(".")]
                if sum(m.file_size for m in members) > MAX_ARCHIVE_BYTES:
                    raise ValueError("ZIP demasiado grande para este notebook inicial.")
                for member in members:
                    key = filename + "::" + member.filename
                    inventory.append({"archivo": key, "bytes": member.file_size})
                    total_bytes += member.file_size
                    if member.file_size > MAX_FILE_BYTES or total_bytes > MAX_ARCHIVE_BYTES:
                        raise ValueError("Se superó el límite de carga configurado.")
                    if PurePosixPath(member.filename).suffix.lower() in {".csv", ".json", ".md"}:
                        contents[key] = archive.read(member)
        else:
            total_bytes += len(payload)
            if len(payload) > MAX_FILE_BYTES or total_bytes > MAX_ARCHIVE_BYTES:
                raise ValueError("Se superó el límite de carga configurado.")
            inventory.append({"archivo": filename, "bytes": len(payload)})
            contents[filename] = payload
    # No se extraen rutas del ZIP al disco ni se ejecuta su contenido.
    return contents, pd.DataFrame(inventory)


def audit_frame(name, frame):
    reports = {}
    findings = []
    def flag(code, count, meaning):
        if count:
            findings.append({"archivo": name, "regla": code,
                             "filas_o_columnas_afectadas": int(count), "interpretacion": meaning})
    def samples(code, mask):
        if bool(mask.any()):
            sample = frame.loc[mask].head(30).copy()
            sample.insert(0, "fila_datos_1_based", sample.index + 1)
            reports["ejemplos_" + code] = sample

    clean = frame.apply(lambda s: s.astype("string").str.strip())
    missing = clean.apply(lambda s: s.str.lower().isin(NA_MARKERS) | s.isna())
    reports["columnas"] = pd.DataFrame({
        "columna": frame.columns,
        "tipo_lectura": [str(t) for t in frame.dtypes],
        "faltantes": missing.sum().to_numpy(),
        "porcentaje_faltante": (missing.mean() * 100).round(2).to_numpy(),
        "valores_distintos_sin_faltantes": [
            int(clean[c].mask(missing[c]).nunique()) for c in frame.columns],
    })
    duplicate_rows = frame.duplicated(keep=False)
    flag("filas_identicas", duplicate_rows.sum(), "Revisar; no eliminar automáticamente.")
    samples("filas_identicas", duplicate_rows)
    flag("columnas_vacias", missing.all().sum(), "Columna sin valores observados.")
    flag("columnas_constantes", (reports["columnas"]["valores_distintos_sin_faltantes"] == 1).sum(),
         "Puede ser una constante legítima; consultar diccionario.")

    fips_col = next((c for c in ["county_fips", "fips", "geoid"] if c in frame), None)
    year_col = "year" if "year" in frame else None
    summary = {"archivo": name, "filas": len(frame), "columnas": len(frame.columns),
               "memoria_mb": round(frame.memory_usage(deep=True).sum() / 1024**2, 2),
               "fips_unicos": None, "ano_min": None, "ano_max": None,
               "filas_con_clave_duplicada": None}
    if fips_col:
        fips = clean[fips_col]
        invalid = (~fips.str.fullmatch(r"[0-9]{5}", na=False)) | missing[fips_col]
        aggregate = fips.str.fullmatch(r"[0-9]{2}000", na=False)
        summary["fips_unicos"] = int(fips.mask(missing[fips_col]).nunique())
        flag("fips_formato", invalid.sum(), "FIPS ausente o con formato distinto a cinco dígitos.")
        flag("posible_agregado_estatal", aggregate.sum(), "Código terminado en 000: verificar unidad geográfica.")
        samples("fips_formato", invalid)
        samples("posible_agregado_estatal", aggregate)
    if year_col:
        years = pd.to_numeric(clean[year_col], errors="coerce")
        valid = years.notna() & years.between(1900, 2100) & years.mod(1).eq(0)
        flag("ano_ausente_o_invalido", (~valid).sum(), "Revisar formato, faltantes y rango 1900–2100.")
        samples("ano_ausente_o_invalido", ~valid)
        if valid.any():
            summary["ano_min"] = int(years[valid].min())
            summary["ano_max"] = int(years[valid].max())
            coverage = pd.DataFrame({"ano": years[valid].astype(int)})
            if fips_col:
                coverage["fips"] = clean.loc[valid, fips_col]
                coverage = coverage.groupby("ano").agg(filas=("fips", "size"),
                                                       claves_geograficas=("fips", "nunique")).reset_index()
            else:
                coverage = coverage.groupby("ano").size().reset_index(name="filas")
            reports["cobertura_anual"] = coverage
    if fips_col and year_col:
        # Son claves candidatas. Confirmar la granularidad en el diccionario.
        comparable_keys = clean[[fips_col, year_col]].mask(missing[[fips_col, year_col]])
        duplicate_keys = comparable_keys.duplicated(keep=False)
        summary["filas_con_clave_duplicada"] = int(duplicate_keys.sum())
        flag("clave_condado_ano_repetida", duplicate_keys.sum(),
             "Verificar granularidad: puede requerir agregación, no eliminación.")
        samples("clave_condado_ano_repetida", duplicate_keys)
        flag("claves_ausentes", missing[[fips_col, year_col]].any(axis=1).sum(),
             "No se puede unir de forma fiable sin revisar la clave.")

    numeric = []
    for col in frame.columns:
        if col.endswith(("_count", "_usd", "_percentile", "_score")):
            values = pd.to_numeric(clean[col].mask(missing[col]), errors="coerce")
            bad = values.isna() & ~missing[col]
            negative = values.lt(0).fillna(False)
            flag("numero_invalido:" + col, bad.sum(), "Valor no numérico en columna candidata.")
            flag("negativo:" + col, negative.sum(), "Consultar reglas de la fuente antes de corregir.")
            samples("numero_invalido_" + col, bad)
            samples("negativo_" + col, negative)
            numeric.append({"columna": col, "min": values.min(), "mediana": values.median(),
                            "max": values.max(), "no_numericos": int(bad.sum())})
    reports["numericas"] = pd.DataFrame(numeric)
    if "flood_related_event_count" in frame:
        flag("definicion_objetivo", 1,
             "No asumir que flood_related_event_count cuenta solo inundaciones: revisar su definición.")
    return summary, pd.DataFrame(findings), reports


## 4. Leer CSV y generar el diagnóstico

Los códigos se conservan como texto. Los marcadores de ausencia se cuentan para el reporte sin reemplazar los valores originales. Los archivos con errores se reportan; no se descartan líneas silenciosamente.

In [ ]:
contents, inventory = unpack_inputs(uploaded)
display(inventory)
frames = {}
load_errors = []
documents = {}
for name, payload in contents.items():
    suffix = PurePosixPath(name).suffix.lower()
    if suffix == ".csv":
        try:
            # No se saltan líneas erróneas ni se cambian valores automáticamente.
            frame = pd.read_csv(io.BytesIO(payload), dtype="string", keep_default_na=False,
                                encoding="utf-8-sig")
            if frame.empty:
                print("CSV vacío:", name)
            frames[name] = frame
        except Exception as exc:
            load_errors.append({"archivo": name, "error": str(exc)})
    elif suffix in {".json", ".md"}:
        documents[name] = payload.decode("utf-8-sig", errors="replace")

if not frames:
    raise ValueError("No se leyó ningún CSV. Revisa los archivos y los errores de carga.")
print("CSV cargados:", len(frames))
print("Los Parquet se omiten en esta primera revisión para no duplicar el CSV equivalente.")
if load_errors:
    print("ATENCIÓN: algunos archivos no se analizaron:")
    display(pd.DataFrame(load_errors))

summaries, findings_parts, details = [], [], {}
for name, frame in frames.items():
    summary, findings, reports = audit_frame(name, frame)
    summaries.append(summary)
    if not findings.empty:
        findings_parts.append(findings)
    details[name] = reports
summary_table = pd.DataFrame(summaries)
findings_table = pd.concat(findings_parts, ignore_index=True) if findings_parts else pd.DataFrame(
    columns=["archivo", "regla", "filas_o_columnas_afectadas", "interpretacion"])
print("RESUMEN GENERAL")
display(summary_table)
print("HALLAZGOS QUE NECESITAN REVISIÓN HUMANA")
display(findings_table)


## 5. Revisar el detalle de una tabla

La variable `selected` permite elegir otro nombre del inventario. Los posibles agregados estatales son advertencias, no una validación contra un catálogo oficial.

In [ ]:
# Cambia este nombre si quieres inspeccionar otro archivo.
selected = next((name for name in frames if name.endswith("county_year_climate_flood_risk.csv")),
                next(iter(frames)))
print("Archivo seleccionado:", selected)
print("Columnas ordenadas por porcentaje de faltantes")
display(details[selected]["columnas"].sort_values("porcentaje_faltante", ascending=False))
print("Cobertura anual: un año con pocas filas NO prueba por sí solo que esté incompleto.")
display(details[selected].get("cobertura_anual", pd.DataFrame()))
print("Resumen de columnas numéricas candidatas")
display(details[selected]["numericas"])
print("Muestras de casos detectados (máximo 30 por regla)")
for rule, table in details[selected].items():
    if rule.startswith("ejemplos_"):
        print(rule)
        display(table)


## 6. Comparar cobertura entre fuentes

In [ ]:
# Comparación de claves entre tablas condado-año; no presupone que deban tener la misma cobertura.
key_sets = {}
for name, frame in frames.items():
    if {"county_fips", "year"}.issubset(frame.columns):
        keys = frame[["county_fips", "year"]].apply(lambda s: s.str.strip())
        valid = keys["county_fips"].str.fullmatch(r"[0-9]{5}", na=False)
        valid &= keys["year"].str.fullmatch(r"[0-9]{4}", na=False)
        key_sets[name] = set(keys.loc[valid].itertuples(index=False, name=None))
comparison_rows = []
names = list(key_sets)
for i, left in enumerate(names):
    for right in names[i + 1:]:
        a, b = key_sets[left], key_sets[right]
        comparison_rows.append({"tabla_a": left, "tabla_b": right,
                                "claves_comunes": len(a & b),
                                "solo_a": len(a - b), "solo_b": len(b - a)})
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table)
print("Diferencias pueden ser legítimas: documentarlas antes de unir o rellenar con ceros.")
print("Documentos de referencia cargados:", list(documents))


## 7. Descargar los reportes

In [ ]:
# Exporta únicamente reportes; no modifica ni vuelve a exportar los datasets originales.
buffer = io.BytesIO()
with zipfile.ZipFile(buffer, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    def add_csv(name, table):
        archive.writestr(name, table.to_csv(index=False).encode("utf-8-sig"))
    add_csv("inventario.csv", inventory)
    add_csv("resumen.csv", summary_table)
    add_csv("hallazgos.csv", findings_table)
    add_csv("errores_carga.csv", pd.DataFrame(load_errors))
    add_csv("comparacion_claves.csv", comparison_table)
    index_rows = []
    for number, (name, reports) in enumerate(details.items(), start=1):
        folder = f"tabla_{number:02d}"
        index_rows.append({"carpeta": folder, "archivo_original": name})
        for report, table in reports.items():
            add_csv(f"{folder}/{report}.csv", table)
    add_csv("indice_tablas.csv", pd.DataFrame(index_rows))
    archive.writestr("LEEME.txt",
        "Auditoría estructural inicial de FloodRisk. No es validación de un modelo.\n"
        "Los hallazgos requieren interpretación; no se corrigieron ni eliminaron filas.\n"
        "El rango de años no identifica el último mes cubierto ni la fecha de publicación.\n"
        "Faltan catálogo oficial FIPS, definición de etiqueta y disponibilidad histórica.\n")
output = "FloodRisk_auditoria_inicial.zip"
with open(output, "wb") as destination:
    destination.write(buffer.getvalue())
files.download(output)


## 8. Decisiones que todavía debemos tomar

- Confirmar unidades geográficas con un catálogo oficial, incluyendo cambios de límites y equivalentes de condado.
- Confirmar qué tipos de evento representan la etiqueta de inundación.
- Distinguir cero eventos de falta de cobertura; no rellenar vacíos automáticamente.
- Verificar la cobertura mensual y los retrasos de publicación; `year` solo permite ver años.
- Revisar el diccionario y las excepciones antes de limpiar.
- Definir el corte temporal y la separación por años antes de entrenar.

**Un reporte sin advertencias no demuestra que los datos estén listos para el modelo.** Es la primera parte de la auditoría.

Referencias: [entrada y salida en Colab](https://colab.research.google.com/notebooks/io.ipynb), [pandas.read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).